# RUL Predictor — Full Notebook
Run cells top to bottom. Do not skip any cell.
**Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# CELL 1 — GPU check + install
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
# Must show T4 GPU. If blank: Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install -q wandb scikit-learn matplotlib pandas

In [ ]:
# CELL 2 — Imports
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from dataclasses import dataclass
import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
# Must print: Using device: cuda

In [ ]:
# CELL 3 — Download NASA CMAPSS data
os.makedirs('data', exist_ok=True)

BASE = 'https://raw.githubusercontent.com/schweryjonas/predictive-maintenance/master/CMAPSSData/'
FILES = ['train_FD001.txt', 'test_FD001.txt', 'RUL_FD001.txt']

for f in FILES:
    dest = f'data/{f}'
    if not os.path.exists(dest):
        print(f'Downloading {f}...')
        urllib.request.urlretrieve(BASE + f, dest)
    else:
        print(f'Already exists: {f}')

print('All files ready.')
# Expected:
# Downloading train_FD001.txt...
# Downloading test_FD001.txt...
# Downloading RUL_FD001.txt...
# All files ready.

In [ ]:
# CELL 4 — Data loading + preprocessing (fixed RUL cap bug)

RUL_CAP = 125
SEQ_LEN = 30
ACTIVE_SENSORS = ['s2','s3','s4','s7','s8','s9',
                  's11','s12','s13','s14','s15','s17','s20','s21']

INDEX_COLS   = ['unit_id', 'cycle']
SETTING_COLS = ['setting_1', 'setting_2', 'setting_3']
SENSOR_COLS  = [f's{i}' for i in range(1, 22)]
ALL_COLS     = INDEX_COLS + SETTING_COLS + SENSOR_COLS

def load_and_prepare(data_dir='data', subset='FD001'):
    dp = lambda f: f'{data_dir}/{f}_{subset}.txt'
    train_df = pd.read_csv(dp('train'), sep=r'\s+', header=None, names=ALL_COLS)
    test_df  = pd.read_csv(dp('test'),  sep=r'\s+', header=None, names=ALL_COLS)
    rul_true = pd.read_csv(dp('RUL'),   sep=r'\s+', header=None, names=['RUL'])

    # Training RUL — piecewise linear capped at 125
    mx = train_df.groupby('unit_id')['cycle'].max().reset_index()
    mx.columns = ['unit_id', 'max_cycle']
    train_df = train_df.merge(mx, on='unit_id')
    train_df['RUL'] = (train_df['max_cycle'] - train_df['cycle']).clip(upper=RUL_CAP)
    train_df.drop('max_cycle', axis=1, inplace=True)

    # FIX: cap test RUL too — same scale as training
    rul_true['RUL'] = rul_true['RUL'].clip(upper=RUL_CAP)
    rul_true['unit_id'] = range(1, len(rul_true) + 1)

    return train_df, test_df, dict(zip(rul_true['unit_id'], rul_true['RUL']))

train_df, test_df, test_rul_map = load_and_prepare()

print(f'Train rows: {len(train_df)} | Unique engines: {train_df["unit_id"].nunique()}')
print(f'Test rows:  {len(test_df)}  | Unique engines: {test_df["unit_id"].nunique()}')
print(f'Test RUL range: {min(test_rul_map.values()):.0f} to {max(test_rul_map.values()):.0f}  ← must be 0–125')
# Expected:
# Train rows: 20631 | Unique engines: 100
# Test rows:  13096  | Unique engines: 100
# Test RUL range: 2 to 125  ← must be 0–125

In [ ]:
# CELL 5 — Dataset class

class CMAPSSDataset(Dataset):
    def __init__(self, df, rul_map, seq_len, sensors, scaler=None, mode='train'):
        self.seq_len = seq_len
        self.mode    = mode
        df = df.copy()

        # Fit or apply scaler
        if scaler is None:
            self.scaler = MinMaxScaler()
            df[sensors] = self.scaler.fit_transform(df[sensors])
        else:
            self.scaler = scaler
            df[sensors] = self.scaler.transform(df[sensors])

        # Health index — weighted average of sensors correlated with RUL
        if 'RUL' in df.columns:
            corr = df[sensors].corrwith(df['RUL']).abs()
            w    = corr / corr.sum()
            df['health_index'] = (df[sensors] * w).sum(axis=1)
        else:
            df['health_index'] = df[sensors].mean(axis=1)

        self.feature_cols = sensors + ['health_index']
        self.seqs, self.labels = self._build(df, rul_map)

    def _build(self, df, rul_map):
        seqs, labels = [], []
        for uid, group in df.groupby('unit_id'):
            vals = group[self.feature_cols].values.astype(np.float32)

            if self.mode == 'train':
                rul_vals = group['RUL'].values.astype(np.float32)
                for i in range(len(vals) - self.seq_len + 1):
                    seqs.append(vals[i:i + self.seq_len])
                    labels.append(rul_vals[i + self.seq_len - 1])
            else:
                # Test: last seq_len cycles, pad if engine has fewer
                if len(vals) >= self.seq_len:
                    seq = vals[-self.seq_len:]
                else:
                    pad = np.zeros((self.seq_len - len(vals), vals.shape[1]), np.float32)
                    seq = np.vstack([pad, vals])
                seqs.append(seq)
                labels.append(float(rul_map[uid]))

        return (torch.tensor(np.array(seqs), dtype=torch.float32),
                torch.tensor(np.array(labels), dtype=torch.float32))

    def __len__(self):          return len(self.seqs)
    def __getitem__(self, i):   return self.seqs[i], self.labels[i]


train_ds = CMAPSSDataset(train_df, None,        SEQ_LEN, ACTIVE_SENSORS, mode='train')
test_ds  = CMAPSSDataset(test_df,  test_rul_map, SEQ_LEN, ACTIVE_SENSORS,
                          scaler=train_ds.scaler, mode='test')

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

n_features = len(train_ds.feature_cols)
print(f'Train samples : {len(train_ds)}')
print(f'Test  samples : {len(test_ds)}')
print(f'n_features    : {n_features}')
print(f'Test RUL range: {test_ds.labels.min().item():.1f} to {test_ds.labels.max().item():.1f}  ← must be 0–125')
# Expected:
# Train samples : 17431
# Test  samples : 100
# n_features    : 15
# Test RUL range: 2.0 to 125.0  ← must be 0–125

In [ ]:
# CELL 6 — Model definitions

@dataclass
class ModelConfig:
    n_features : int
    seq_len    : int
    hidden_dim : int = 128
    n_layers   : int = 2
    dropout    : float = 0.2


class LSTMForecaster(nn.Module):
    """LSTM → Transformer attention → regression head"""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=cfg.n_features,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.n_layers,
            batch_first=True,
            dropout=cfg.dropout
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.hidden_dim, nhead=4,
            dim_feedforward=256, dropout=cfg.dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.head = nn.Sequential(
            nn.Linear(cfg.hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = self.transformer(out)
        return self.head(out.mean(dim=1)).squeeze(-1)


class CNN1DForecaster(nn.Module):
    """1D CNN → global avg pool → regression head"""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(cfg.n_features, 64,  kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(64,             128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(128,            cfg.hidden_dim, kernel_size=3, padding=1), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(cfg.hidden_dim, 64), nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (B, T, F) → (B, F, T) for Conv1d
        out = self.encoder(x.permute(0, 2, 1))
        return self.head(out.mean(dim=-1)).squeeze(-1)


# Quick sanity check
cfg   = ModelConfig(n_features=n_features, seq_len=SEQ_LEN)
dummy = torch.randn(4, SEQ_LEN, n_features)
print('LSTM  output:', LSTMForecaster(cfg)(dummy).shape)   # torch.Size([4])
print('CNN1D output:', CNN1DForecaster(cfg)(dummy).shape)  # torch.Size([4])

In [ ]:
# CELL 7 — train_model function
# This cell must run before Cells 8 and 9.

def eval_model(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds.append(model(xb).cpu())
            targets.append(yb)
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    rmse = float(np.sqrt(np.mean((preds - targets) ** 2)))
    mae  = float(np.mean(np.abs(preds - targets)))
    return rmse, mae, preds, targets


def train_model(model, train_loader, test_loader,
                epochs=200, lr=3e-4, run_name='run'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()

    history = {'train_rmse': [], 'test_rmse': [], 'test_mae': []}
    best_test_rmse = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        scheduler.step()

        train_rmse = float(np.sqrt(np.mean(train_losses)))
        test_rmse, test_mae, _, _ = eval_model(model, test_loader)

        history['train_rmse'].append(train_rmse)
        history['test_rmse'].append(test_rmse)
        history['test_mae'].append(test_mae)

        wandb.log({'train_rmse': train_rmse,
                   'test_rmse':  test_rmse,
                   'test_mae':   test_mae}, step=epoch)

        if test_rmse < best_test_rmse:
            best_test_rmse = test_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 20 == 0:
            print(f'  Epoch {epoch:3d} | train_rmse: {train_rmse:.2f} | '
                  f'test_rmse: {test_rmse:.2f} | test_mae: {test_mae:.2f}')

    # Restore best weights
    model.load_state_dict(best_state)
    print(f'  Best test RMSE: {best_test_rmse:.2f}')
    return model, history

print('train_model defined. Ready to train.')

In [ ]:
# CELL 8 — W&B login + train LSTM
import wandb
wandb.login()  # paste your key from wandb.ai/authorize

wandb.init(project='rul-predictor', name='lstm-transformer', reinit=True)

cfg        = ModelConfig(n_features=n_features, seq_len=SEQ_LEN, hidden_dim=128, n_layers=2, dropout=0.2)
lstm_model = LSTMForecaster(cfg).to(device)

print(f'LSTM params: {sum(p.numel() for p in lstm_model.parameters()):,}')
print('Training LSTM — 200 epochs, prints every 20...')
lstm_model, lstm_history = train_model(
    lstm_model, train_loader, test_loader, epochs=200, lr=3e-4, run_name='lstm'
)
wandb.finish()

# Expected final test RMSE: 14–20 cycles
# Training time on T4: ~12–18 minutes

In [ ]:
# CELL 9 — Train CNN
wandb.init(project='rul-predictor', name='cnn1d', reinit=True)

cnn_model = CNN1DForecaster(cfg).to(device)
print(f'CNN params: {sum(p.numel() for p in cnn_model.parameters()):,}')
print('Training CNN1D — 200 epochs...')
cnn_model, cnn_history = train_model(
    cnn_model, train_loader, test_loader, epochs=200, lr=3e-4, run_name='cnn'
)
wandb.finish()
# Training time: ~6–8 minutes

In [ ]:
# CELL 10 — Evaluate both models + generate plots for README

lstm_rmse, lstm_mae, lstm_preds, targets = eval_model(lstm_model, test_loader)
cnn_rmse,  cnn_mae,  cnn_preds,  _       = eval_model(cnn_model,  test_loader)

print('=== FINAL RESULTS ===')
print(f'LSTM  — RMSE: {lstm_rmse:.2f} | MAE: {lstm_mae:.2f}')
print(f'CNN1D — RMSE: {cnn_rmse:.2f}  | MAE: {cnn_mae:.2f}')

# Pick better model
best_preds = lstm_preds if lstm_rmse < cnn_rmse else cnn_preds
best_name  = 'LSTM' if lstm_rmse < cnn_rmse else 'CNN1D'
print(f'Winner: {best_name}')

errors = best_preds - targets

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'RUL Predictor — {best_name} | RMSE={min(lstm_rmse,cnn_rmse):.2f}', fontsize=13)

# Plot 1: Predicted vs Actual
axes[0].scatter(targets, best_preds, alpha=0.7, s=25, c='steelblue')
axes[0].plot([0, 125], [0, 125], 'r--', linewidth=1)
axes[0].set_xlabel('Actual RUL (cycles)')
axes[0].set_ylabel('Predicted RUL (cycles)')
axes[0].set_title('Predicted vs Actual')

# Plot 2: Error distribution
axes[1].hist(errors, bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Prediction Error (cycles)')
axes[1].set_title('Error Distribution')

# Plot 3: LSTM vs CNN training curves
axes[2].plot(lstm_history['test_rmse'], label='LSTM test RMSE')
axes[2].plot(cnn_history['test_rmse'],  label='CNN test RMSE')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('RMSE')
axes[2].set_title('Training Curves')
axes[2].legend()

plt.tight_layout()
plt.savefig('eval_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as eval_results.png — download and put in your README')

In [ ]:
# CELL 11 — Save models + download
torch.save(lstm_model.state_dict(), 'lstm_rul.pth')
torch.save(cnn_model.state_dict(),  'cnn_rul.pth')
print('Models saved.')

# Download both to your machine
from google.colab import files
files.download('lstm_rul.pth')
files.download('cnn_rul.pth')
files.download('eval_results.png')
print('Downloaded. Put pth files in your repo under checkpoints/')